#### Imports

In [6]:
from pybaseball import statcast
import pandas as pd
import numpy as np

#### Collect Data

In [2]:
# Pull full 2024 season
data = statcast('2024-03-20', '2024-09-29')

This is a large query, it may take a moment to complete


100%|██████████| 194/194 [01:35<00:00,  2.04it/s]


In [ ]:
# Save to CSV as to not have to pull each time
data.to_csv(r'F:\statcast-pitch-analysis\Data\statcast_2024.csv', index=False)
print(data.shape)

<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
C:\Users\socce\AppData\Local\Temp\ipykernel_27280\340454138.py:2: SyntaxWarning: invalid escape sequence '\s'
  data.to_csv('F:\statcast-pitch-analysis\Data\statcast_2024.csv', index=False)


(731905, 119)


#### Cleanup

In [ ]:
df = pd.read_csv(r'F:\statcast-pitch-analysis\Data\statcast_2024.csv', low_memory=False)

# Keep only the columns to be used
cols = ['pitcher', 'player_name', 'pitch_type', 'pitch_name', 'release_speed', 'release_spin_rate', 'pfx_x', 'pfx_z','p_throws' ]
df = df[cols]

# Drop rows with NA's
df = df.dropna(subset=['pfx_x', 'pfx_z', 'release_speed', 'pitch_type'])

# See what pitch types exist and how common they are
print(df['pitch_type'].value_counts())
print(f'\nShape after cleaning: {df.shape}')

pitch_type
FF    230001
SI    113750
SL    106729
CH     73720
FC     58787
ST     52879
CU     44030
FS     22209
KC     12879
SV      3724
KN      1057
FA       631
EP       575
FO       170
SC       155
CS        62
PO        55
UN         2
Name: count, dtype: int64

Shape after cleaning: (721415, 9)


In [11]:
#Drop rare/uninportant pitches
keep = ['FF', 'SI', 'SL', 'CH', 'FC', 'ST', 'CU', 'FS', 'KC', 'SV']
df = df[df['pitch_type'].isin(keep)]

print(df['pitch_type'].value_counts())
print(f'\nFinal shape: {df.shape}')

pitch_type
FF    230001
SI    113750
SL    106729
CH     73720
FC     58787
ST     52879
CU     44030
FS     22209
KC     12879
SV      3724
Name: count, dtype: int64

Final shape: (718708, 9)


In [13]:
# Mirror pfx_x for LHP so all pitchers are in the same reference frame
df['pfx_x_adj'] = df.apply(
    lambda row: -row['pfx_x'] if row['p_throws'] == 'L' else row['pfx_x'],
    axis=1
)

# Save to ne CSV
df.to_csv(r'F:\statcast-pitch-analysis\Data\statcast_2024_CleanAndMirrored.csv', index=False)